# Hands-on 3: Mengirim Data Sensor ke MQTT

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* oleh Onno W. Purbo (Halaman 73–78).

### Tujuan Hands-on:
1. Memahami arsitektur *hub-and-spoke* MQTT untuk pengumpulan data sensor (telemetri).
2. Membangun publisher / simulator sensor IoT menggunakan pustaka `paho-mqtt` di Python.
3. Mengirimkan payload JSON terstruktur ke topik MQTT bertingkat secara periodik.
4. Menerapkan variasi realistis untuk kasus smart room dan greenhouse (pertanian presisi).


### 1. Konfigurasi Broker dan Pustaka Paho-MQTT
Dalam latihan ini, klien publisher akan terhubung ke broker Mosquitto di `localhost:1883`.


In [1]:
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt

BROKER_HOST = "localhost"
BROKER_PORT = 1883
TOPIC_ROOM = "kampus/lab1/room01/telemetry"
CLIENT_ID = "notebook-room-publisher"

print(f"[CONFIG] Broker: {BROKER_HOST}:{BROKER_PORT}")
print(f"[CONFIG] Topik : {TOPIC_ROOM}")


[CONFIG] Broker: localhost:1883
[CONFIG] Topik : kampus/lab1/room01/telemetry


### 2. Generator Data Sensor Sintetis (Room Telemetry)
Fungsi membangkitkan data suhu, kelembaban, CO2, daya listrik, dan timestamp ISO-8601.


In [2]:
def generate_sensor_data(device_id="room01"):
    return {
        "device_id": device_id,
        "temperature_c": round(random.uniform(24.0, 30.0), 2),
        "humidity_pct": round(random.uniform(45.0, 65.0), 2),
        "co2_ppm": random.randint(450, 900),
        "power_watt": round(random.uniform(100.0, 160.0), 2),
        "timestamp": datetime.now().isoformat(timespec="seconds")
    }

contoh_payload = generate_sensor_data()
print("Contoh Payload JSON:")
print(json.dumps(contoh_payload, indent=2))


Contoh Payload JSON:
{
  "device_id": "room01",
  "temperature_c": 29.11,
  "humidity_pct": 55.55,
  "co2_ppm": 654,
  "power_watt": 155.1,
  "timestamp": "2026-10-21T22:03:55"
}


### 3. Eksekusi Pengiriman Pesan ke Broker MQTT
Membuka koneksi ke Mosquitto dan mempublikasikan 5 paket data secara periodik.


In [3]:
client = mqtt.Client(client_id=CLIENT_ID)

try:
    client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    print(f"[INFO] Terhubung ke broker MQTT ({BROKER_HOST}:{BROKER_PORT})")

    for i in range(5):
        payload = generate_sensor_data()
        payload_json = json.dumps(payload)
        res = client.publish(TOPIC_ROOM, payload_json, qos=1)
        res.wait_for_publish(timeout=3)
        print(f"[{i+1}/5] Publish sukses ke {TOPIC_ROOM} -> {payload_json}")
        time.sleep(1)

    client.disconnect()
    print("[SELESAI] Pengiriman telemetri ruang selesai.")
except Exception as e:
    print(f"[ERROR] Koneksi MQTT gagal: {e}")


[INFO] Terhubung ke broker MQTT (localhost:1883)


[1/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 27.39, "humidity_pct": 58.53, "co2_ppm": 733, "power_watt": 134.86, "timestamp": "2026-10-21T22:03:55"}


[2/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 26.54, "humidity_pct": 47.23, "co2_ppm": 497, "power_watt": 135.55, "timestamp": "2026-10-21T22:03:59"}


[3/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 26.26, "humidity_pct": 63.37, "co2_ppm": 870, "power_watt": 120.26, "timestamp": "2026-10-21T22:04:03"}


[4/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 28.16, "humidity_pct": 60.07, "co2_ppm": 802, "power_watt": 112.9, "timestamp": "2026-10-21T22:04:07"}


[5/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 27.32, "humidity_pct": 52.32, "co2_ppm": 853, "power_watt": 138.99, "timestamp": "2026-10-21T22:04:11"}


[SELESAI] Pengiriman telemetri ruang selesai.


### 4. Variasi Realistis: Sensor Pertanian Presisi (Greenhouse)
Mengirimkan parameter pertanian: suhu udara, kelembaban tanah (*soil moisture*), dan intensitas cahaya (*lux*).


In [4]:
TOPIC_GREENHOUSE = "kampus/greenhouse/zone1/telemetry"

def generate_greenhouse_data():
    return {
        "device_id": "greenhouse-zone1",
        "air_temp_c": round(random.uniform(25.0, 35.0), 2),
        "soil_moisture_pct": round(random.uniform(30.0, 80.0), 2),
        "light_lux": random.randint(8000, 30000),
        "timestamp": datetime.now().isoformat(timespec="seconds")
    }

client_gh = mqtt.Client(client_id="notebook-gh-pub")

try:
    client_gh.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    for i in range(3):
        gh_data = generate_greenhouse_data()
        gh_json = json.dumps(gh_data)
        client_gh.publish(TOPIC_GREENHOUSE, gh_json, qos=1)
        print(f"[GREENHOUSE {i+1}/3] {gh_json}")
        time.sleep(1)
    client_gh.disconnect()
    print("[SELESAI] Telemetri greenhouse terkirim.")
except Exception as e:
    print(f"[ERROR] Koneksi MQTT greenhouse gagal: {e}")


[GREENHOUSE 1/3] {"device_id": "greenhouse-zone1", "air_temp_c": 29.19, "soil_moisture_pct": 76.37, "light_lux": 18480, "timestamp": "2026-10-21T22:04:15"}


[GREENHOUSE 2/3] {"device_id": "greenhouse-zone1", "air_temp_c": 33.02, "soil_moisture_pct": 70.32, "light_lux": 18188, "timestamp": "2026-10-21T22:04:16"}


[GREENHOUSE 3/3] {"device_id": "greenhouse-zone1", "air_temp_c": 27.56, "soil_moisture_pct": 59.85, "light_lux": 26259, "timestamp": "2026-10-21T22:04:17"}


[SELESAI] Telemetri greenhouse terkirim.


## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 3 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 10. Variasi realistis: publisher untuk sensor greenhouse

Buat file scripts/mqtt/publish_greenhouse.py:
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt
BROKER_HOST = "localhost"
BROKER_PORT = 1883
TOPIC = "kampus/greenhouse/zone1/telemetry"
CLIENT_ID = "publisher-greenhouse-zone1"
client = mqtt.Client(client_id=CLIENT_ID)
def generate_data():
return {
"device_id": "greenhouse-zone1",
"air_temp_c": round(random.uniform(25.0, 35.0), 2),
"soil_moisture_pct": round(random.uniform(30.0, 80.0), 2),
"light_lux": random.randint(8000, 30000),
"timestamp": datetime.now().isoformat(timespec="seconds")
}
client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
while True:
payload = generate_data()
payload_json = json.dumps(payload)
client.publish(TOPIC, payload_json, qos=1)
print(f"[GREENHOUSE] {payload_json}")
time.sleep(10)

Ini relevan untuk kasus pertanian presisi, di mana sensor tanah dan cuaca lokal
dipakai untuk optimasi irigasi dan pemantauan kondisi tanaman.

### 11. Langkah kerja yang harus dilakukan pembaca

Tahap

Kegiatan

Pasang dan aktifkan Mosquitto di Ubuntu.

Uji broker dengan mosquitto_pub dan mosquitto_sub.

Pasang paho-mqtt di virtual environment.

Buat publisher Python sederhana.

Kirim data JSON secara periodik.

Amati pesan yang masuk dengan tool terminal atau subscriber
Python.

### 12. Tantangan kecil yang menantang

Agar latihan terasa fun tetapi tetap menantang, pembaca bisa mencoba:
- ganti localhost dengan IP Raspberry Pi atau server Ubuntu lain
- ubah topik menjadi beberapa ruangan: room01, room02, room03
- kirim data dengan interval berbeda
- tambahkan field battery_pct
- kirim status perangkat ke topik status
- buat satu publisher untuk suhu, satu lagi untuk energi